# Feature Engineering with OHLCV 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
import yfinance as yf
import warnings

# Suppress fragmentation warnings (many column inserts are fine for our use case)
# Suppress fragmentation warnings (many column inserts are fine for our use case)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

In [2]:
# Show all columns
pd.set_option('display.max_columns', 150)
pd.set_option('display.width', 100)

# Load OHLCV data
ohlcv_df = pd.read_csv('stock_ohlcv_data.csv')
print(f"Loaded OHLCV data: {ohlcv_df.shape}")
print(f"Unique tickers: {ohlcv_df['ticker'].nunique()}")
display(ohlcv_df.head())

# Group by ticker into dictionary for efficient iteration
ticker_data = {ticker: group.copy().set_index('date').sort_index() 
               for ticker, group in ohlcv_df.groupby('ticker')}

# Convert all indices to datetime
for ticker in ticker_data:
    ticker_data[ticker].index = pd.to_datetime(ticker_data[ticker].index, utc=True)

# ============= Apply Liquidity Filter to Each Ticker =============
def find_liquid_start(df, min_volume_pct=0.98, min_ohlc_pct=0.99, max_days_to_drop=126):
    """
    Find the earliest start date where the stock has:
    - >= min_volume_pct non-zero volume days
    - >= min_ohlc_pct non-zero OHLC values
    from that point forward. Drop up to max_days_to_drop (1 year) from the start.
    """
    for days_to_drop in range(0, min(max_days_to_drop + 1, len(df))):
        trimmed = df.iloc[days_to_drop:]
        if len(trimmed) == 0:
            return None
        
        # Check volume: >= min_volume_pct non-zero
        volume_pct = (trimmed["Volume"] > 0).mean()
        
        # Check OHLC: >= min_ohlc_pct non-zero for each column
        ohlc_pcts = [(trimmed[col] > 0).mean() for col in ["Open", "High", "Low", "Close"]]
        min_ohlc = min(ohlc_pcts)
        
        if volume_pct >= min_volume_pct and min_ohlc >= min_ohlc_pct:
            return trimmed
    
    return None

print(f"\n=== Applying Liquidity Filter (≥98% volume, ≥99% OHLC non-zero) ===")
filtered_ticker_data = {}
dropped_tickers = []

for ticker, df in ticker_data.items():
    result = find_liquid_start(df, min_volume_pct=0.98, min_ohlc_pct=0.99, max_days_to_drop=126)
    if result is not None and len(result) >= 150:  # Need minimum data for features
        filtered_ticker_data[ticker] = result
    else:
        dropped_tickers.append(ticker)

print(f"Tickers before filter: {len(ticker_data)}")
print(f"Tickers after filter: {len(filtered_ticker_data)}")
print(f"Dropped tickers: {len(dropped_tickers)}")
if dropped_tickers:
    display(dropped_tickers)

# Replace ticker_data with filtered version
ticker_data = filtered_ticker_data

# ============= Sort tickers by their first available date =============
# This ensures batches are grouped by IPO date cohorts
ticker_first_dates = {ticker: df.index.min() for ticker, df in ticker_data.items()}
sorted_tickers = sorted(ticker_first_dates.keys(), key=lambda t: ticker_first_dates[t])

print(f"\n=== Tickers Sorted by First Date (after liquidity filter) ===")
print(f"Earliest ticker: {sorted_tickers[0]} ({ticker_first_dates[sorted_tickers[0]].date()})")
print(f"Latest ticker: {sorted_tickers[-1]} ({ticker_first_dates[sorted_tickers[-1]].date()})")

# Show distribution of first dates
first_dates_series = pd.Series(ticker_first_dates)
print(f"\nFirst date distribution:")
print(f"  Min: {first_dates_series.min().date()}")
print(f"  Max: {first_dates_series.max().date()}")
print(f"  Median: {first_dates_series.median().date()}")

Loaded OHLCV data: (5336503, 7)
Unique tickers: 3093


,date,ticker,Open,High,Low,Close,Volume
0,2018-01-02 00:00:00-05:00,BRK-A,297400.0,298000.0,294000.0,295755.0,300
1,2018-01-03 00:00:00-05:00,BRK-A,296200.0,299920.0,295801.0,299905.0,200
2,2018-01-04 00:00:00-05:00,BRK-A,300450.0,302980.0,300000.0,300516.0,500
3,2018-01-05 00:00:00-05:00,BRK-A,302200.0,302430.0,297900.0,301525.0,300
4,2018-01-08 00:00:00-05:00,BRK-A,299500.0,304530.0,299500.0,304180.0,500



=== Applying Liquidity Filter (≥98% volume, ≥99% OHLC non-zero) ===
Tickers before filter: 3093
Tickers after filter: 2910
Dropped tickers: 183


['ABL',
 'ACEL',
 'AEBI',
 'AERO',
 'AFGE',
 'AHCO',
 'AKO-A',
 'ALH',
 'ALRS',
 'ALTI',
 'AMBQ',
 'AMCR',
 'AMRZ',
 'APXT',
 'ARX',
 'ASIC',
 'ATS',
 'AUGO',
 'BBOT',
 'BCAL',
 'BCSS',
 'BETA',
 'BETR',
 'BGM',
 'BGSI',
 'BHRB',
 'BITF',
 'BLLN',
 'BLSH',
 'BMNR',
 'BRBI',
 'BTBT',
 'BTDR',
 'BTQ',
 'BULL',
 'BWLP',
 'BWMX',
 'CAI',
 'CCCX',
 'CCZ',
 'CHYM',
 'CIG-C',
 'CLSK',
 'CMCSV',
 'CNL',
 'CPAC',
 'CRCL',
 'CRML',
 'CTOS',
 'CUBB',
 'DEC',
 'DMII',
 'DMIIU',
 'ECX',
 'EFC-PD',
 'EFXT',
 'EICA',
 'ELVR',
 'EMA',
 'ERO',
 'ESBA',
 'EVO',
 'EXEEL',
 'EXEEW',
 'EXEEZ',
 'FCRX',
 'FER',
 'FERG',
 'FIG',
 'FIGR',
 'FISK',
 'FLUT',
 'FLY',
 'FORTY',
 'FRMI',
 'FSUN',
 'GCMG',
 'GDYN',
 'GEGGL',
 'GIBO',
 'GLXY',
 'GRP-UN',
 'HBNB',
 'HCXY',
 'HDL',
 'HNGE',
 'HTFB',
 'HTFC',
 'HTFL',
 'HYMC',
 'IGIC',
 'IMTX',
 'INDV',
 'IVT',
 'JBS',
 'JCAP',
 'KDK',
 'KEN',
 'KLAR',
 'LGN',
 'LOT',
 'LUNR',
 'LWACU',
 'MAAS',
 'MH',
 'MHLA',
 'MIAX',
 'MICC',
 'MKC-V',
 'MLTX',
 'MNMD',
 'MNTN',
 'M


=== Tickers Sorted by First Date (after liquidity filter) ===
Earliest ticker: A (2018-01-02)
Latest ticker: ETOR (2025-05-13)

First date distribution:
  Min: 2018-01-02
  Max: 2025-05-13
  Median: 2018-01-02


# ETF data pull for beta

In [3]:
START = "2017-01-01"
END   = "2025-12-31"

ETF_LIST = {
    'XLK': 'Technology',
    'XLF': 'Financials',
    'XLE': 'Energy',
    'XLI': 'Industrials',
    'XLY': 'Consumer Discretionary',
    'XLV': 'Healthcare',
    'XLU': 'Utilities',
    'SPHB': 'High Beta',
    'SPLV': 'Low Volatility',
    'IWD': 'Value',
    'IWF': 'Growth',
    'MTUM': 'Momentum',
    'QUAL': 'Quality',
    'IWM': 'Small Cap',
    'TLT': 'Long Rates',
    'SHY': 'Short Rates',
    'TIP': 'Inflation',
    '^VIX': 'Volatility'
}

# Required date range (warmup coverage)
REQUIRED_START = pd.Timestamp("2017-07-01", tz="UTC")
REQUIRED_END   = pd.Timestamp("2025-12-17", tz="UTC")

BETA_WINDOW = 126
MIN_PERIODS = 63
EPS = 1e-10

In [4]:
# --------------------- Download SPY + ETFs in one call ---------------------
ALL_TICKERS = ["SPY"] + list(ETF_LIST.keys())

print("Downloading SPY + factor ETFs...")
close_px = yf.download(
    ALL_TICKERS,
    start=START,
    end=END,
    progress=False
)["Close"]

if isinstance(close_px, pd.Series):
    raise ValueError("yfinance returned a Series; expected a DataFrame of Close prices.")

# Ensure timezone-aware UTC index
close_px.index = pd.to_datetime(close_px.index).tz_localize("UTC")

# Standardize columns to lowercase tickers
close_px.columns = [str(c).lower() for c in close_px.columns]

# --------------------- Compute backward-looking daily log returns (vectorized) ---------------------
returns = np.log(close_px / close_px.shift(1)).dropna()

# --------------------- Coverage filter (drop factors without required history window) ---------------------
def has_required_coverage(ret_col: pd.Series) -> bool:
    window = ret_col.loc[REQUIRED_START:REQUIRED_END]
    return (len(window) > 0) and window.notna().all()

valid_cols = [c for c in returns.columns if has_required_coverage(returns[c])]
discarded_cols = [c for c in returns.columns if c not in valid_cols]

returns = returns[valid_cols].copy()

print("\nReturn Matrix Summary:")
print(f"  Loaded tickers (requested): {len(ALL_TICKERS)} (SPY + {len(ETF_LIST)} ETFs)")
print(f"  Valid tickers (kept):       {len(valid_cols)}")
if discarded_cols:
    print(f"  Discarded tickers:          {len(discarded_cols)} -> {discarded_cols}")
print(f"  Shared dates:               {returns.shape[0]}")
print(f"  Date range:                 {returns.index.min().date()} to {returns.index.max().date()}")

factor_returns = returns.copy()  # columns include 'spy', 'xlk', 'xlf', etc.

/var/folders/2r/rwr3ts2508v73p4rwyk8bgt00000gn/T/ipykernel_13893/2599334402.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  close_px = yf.download(



Return Matrix Summary:
  Loaded tickers (requested): 19 (SPY + 18 ETFs)
  Valid tickers (kept):       19
  Shared dates:               2260
  Date range:                 2017-01-04 to 2025-12-30


# Feature Engineering

In [5]:
def beta_ft(
    stock_df: pd.DataFrame,
    factor_returns_df: pd.DataFrame,
    window: int = 126,
    min_periods: int = 63,
    eps: float = 1e-10
) -> pd.DataFrame:
    """
    Rolling beta of a stock vs multiple factor returns.
    Only overlapping dates between stock and factor calendars are used.
    """

    out = stock_df.copy()

    # Ensure UTC index
    if out.index.tz is None:
        out.index = pd.to_datetime(out.index).tz_localize("UTC")
    else:
        out.index = out.index.tz_convert("UTC")

    # Stock log returns
    stock_ret = np.log(out["Close"] / out["Close"].shift(1))

    # ---- STRICT date intersection (no reindexing) ----
    stock_dates = out.index.normalize()
    common_dates = stock_dates.intersection(factor_returns_df.index)

    if len(common_dates) == 0:
        raise ValueError("No overlapping dates between stock and factor returns.")

    # Filter stock returns
    stock_mask = stock_dates.isin(common_dates)
    stock_ret = stock_ret.loc[stock_mask]

    # Filter factor returns
    factors_aligned = factor_returns_df.loc[common_dates]

    # Restore stock timestamps
    factors_aligned.index = out.index[stock_mask]

    # Rolling variance for factors
    factor_var = factors_aligned.rolling(window, min_periods=min_periods).var()

    # Rolling beta vs each factor
    for col in factors_aligned.columns:
        cov = stock_ret.rolling(window, min_periods=min_periods).cov(factors_aligned[col])
        out.loc[stock_mask, f"beta_{col}"] = cov / (factor_var[col] + eps)

    return out

In [6]:
def ema(series: pd.Series, span: int) -> pd.Series:
    """Exponential moving average."""
    return series.ewm(span=span, adjust=False).mean()


def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    """
    Relative Strength Index (Wilder's smoothing).
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, adjust=False).mean()

    rs = avg_gain / (avg_loss + 1e-10)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def macd(series: pd.Series,
         fast: int = 12,
         slow: int = 26,
         signal: int = 9) -> pd.DataFrame:
    """
    MACD line, signal line, and histogram.
    """
    ema_fast = ema(series, fast)
    ema_slow = ema(series, slow)
    macd_line = ema_fast - ema_slow
    signal_line = ema(macd_line, signal)
    hist = macd_line - signal_line

    return pd.DataFrame({
        "macd": macd_line,
        "macd_signal": signal_line,
        "macd_hist": hist
    })


def atr(high: pd.Series,
        low: pd.Series,
        close: pd.Series,
        window: int = 14) -> pd.Series:
    """
    Average True Range.
    """
    prev_close = close.shift(1)

    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()

    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr_vals = tr.rolling(window=window, min_periods=1).mean()
    return atr_vals


def parkinson_vol(high: pd.Series,
                  low: pd.Series,
                  window: int = 30) -> pd.Series:
    """
    Parkinson volatility estimator (annualization omitted; this is raw rolling vol).
    sigma^2 = (1 / (4 n ln 2)) * sum( ln(H/L)^2 ), then sqrt.
    """
    log_hl = (high / low).replace(0, np.nan).apply(np.log)
    daily_var = (log_hl ** 2) / (4 * np.log(2))
    rolling_var = daily_var.rolling(window=window, min_periods=1).mean()
    return np.sqrt(rolling_var)


def obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """
    On-Balance Volume.
    """
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum()


def roc(series: pd.Series, period: int = 10, eps: float = 1e-10) -> pd.Series:
    """
    Rate of Change: (price_t / price_{t-k}) - 1.
    Uses epsilon to avoid division by zero.
    """
    prev = series.shift(period)
    return series / (prev + eps) - 1


def pct_change(series: pd.Series, period: int = 1, eps: float = 1e-10) -> pd.Series:
    """
    Percentage change: (current - previous) / previous.
    Uses epsilon to avoid division by zero.
    """
    prev = series.shift(period)
    return (series - prev) / (prev + eps)


def stoch_k(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> pd.Series:
    """
    Stochastic %K: measures where close is relative to the high-low range over a period.
    %K = 100 * (Close - Lowest Low) / (Highest High - Lowest Low)
    Already scale-invariant (0-100 range).
    """
    lowest_low = low.rolling(window=window, min_periods=1).min()
    highest_high = high.rolling(window=window, min_periods=1).max()
    
    k = 100 * (close - lowest_low) / (highest_high - lowest_low + 1e-10)
    return k


def stoch_d(high: pd.Series, low: pd.Series, close: pd.Series, 
            k_window: int = 14, d_window: int = 3) -> pd.Series:
    """
    Stochastic %D: smoothed moving average of %K.
    %D = SMA(%K, d_window)
    Already scale-invariant (0-100 range).
    """
    k = stoch_k(high, low, close, window=k_window)
    d = k.rolling(window=d_window, min_periods=1).mean()
    return d


def mfm(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    """
    Money Flow Multiplier.
    MFM = ((Close - Low) - (High - Close)) / (High - Low)
    Scale-invariant: always between -1 and +1.
    +1 = close at high (strong buying pressure)
    -1 = close at low (strong selling pressure)
    0 = close at midpoint
    """
    return ((close - low) - (high - close)) / (high - low + 1e-10)


def ad_oscillator(high: pd.Series, low: pd.Series, close: pd.Series, volume: pd.Series) -> pd.Series:
    """
    Accumulation/Distribution Oscillator (Chaikin A/D Line).
    AD = cumsum( ((Close - Low) - (High - Close)) / (High - Low) * Volume )
    
    The Money Flow Multiplier ((Close - Low) - (High - Close)) / (High - Low) is scale-invariant (-1 to +1).
    But cumulative AD is not scale-invariant, so we return pct_change for features.
    """
    # Money Flow Multiplier: already scale-invariant (-1 to +1)
    money_flow_mult = mfm(high, low, close)
    
    # Money Flow Volume
    mfv = money_flow_mult * volume
    
    # Cumulative A/D Line
    ad_line = mfv.cumsum()
    
    return ad_line

In [7]:
# Feature engineering sections will be implemented below in separate code blocks

In [8]:
# ============= 1. Trend / Momentum Features =============

# Log Returns (more stationary and better for modeling)
def trend_ft(df, ticker_name):
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Skip tickers with insufficient data
    if len(df) < 160:  # Need at least 150 days for EMA 120 + buffer
        return None
    
    # Add ticker column
    df['ticker'] = ticker_name
    
    # Clean OHLC data: replace 0 with NaN and interpolate
    # Zero prices are invalid and likely data gaps
    # Linear interpolation only between valid values (no extrapolation at edges)
    for col in ["Open", "High", "Low", "Close"]:
        df[col] = df[col].replace(0, np.nan).interpolate(method='linear')

    # EMAs
    for span in [3, 5, 10, 21, 63, 126]:
        df[f"ema_{span}"] = ema(df["Close"], span)

    # Distance to EMA (percentage only - more stationary for BTC's wide price range)
    for span in [3, 5, 10, 21, 63, 126]:
        df[f"dist_to_ema_{span}_pct"] = (df["Close"] - df[f"ema_{span}"]) / df[f"ema_{span}"]

    # EMA crossover indicators (shorter EMA above longer EMA)
    # Short-term crossovers
    df["ema3_above_ema5"] = (df["ema_3"] - df["ema_5"]) / df["ema_5"]
    df["ema3_above_ema10"] = (df["ema_3"] - df["ema_10"]) / df["ema_10"]
    df["ema5_above_ema10"] = (df["ema_5"] - df["ema_10"]) / df["ema_10"]
    df["ema5_above_ema21"] = (df["ema_5"] - df["ema_21"]) / df["ema_21"]
    df["ema5_above_ema63"] = (df["ema_5"] - df["ema_63"]) / df["ema_63"]
    # Medium-term crossovers
    df["ema10_above_ema21"] = (df["ema_10"] - df["ema_21"]) / df["ema_21"]
    df["ema10_above_ema63"] = (df["ema_10"] - df["ema_63"]) / df["ema_63"]
    df["ema21_above_ema63"] = (df["ema_21"] - df["ema_63"]) / df["ema_63"]
    df["ema21_above_ema126"] = (df["ema_21"] - df["ema_126"]) / df["ema_126"]
    # Long-term crossover (golden/death cross equivalent)
    df["ema63_above_ema126"] = (df["ema_63"] - df["ema_126"]) / df["ema_126"]

    # MACD - Multiple timescales for different momentum horizons
    # Fast MACD 1 week to 3 weeks
    macd_fast = macd(df["Close"], fast=5, slow=15, signal=5)
    macd_fast.columns = ["macd_fast", "macd_signal_fast", "macd_hist_fast"]
    df = pd.concat([df, macd_fast], axis=1)
    # MACD as % of price (scale-invariant)
    df["macd_fast_pct"] = df["macd_fast"] / df["Close"]
    df["macd_signal_fast_pct"] = df["macd_signal_fast"] / df["Close"]
    df["macd_hist_fast_pct"] = df["macd_hist_fast"] / df["Close"]
    # MACD diff changes
    df["macd_fast_change"] = df["macd_fast"].diff()
    df["macd_hist_fast_change"] = df["macd_hist_fast"].diff()

    # Standard MACD 2 weeks to 1 month
    macd_std = macd(df["Close"], fast=12, slow=26, signal=9)
    macd_std.columns = ["macd_std", "macd_signal_std", "macd_hist_std"]
    df = pd.concat([df, macd_std], axis=1)
    # MACD as % of price (scale-invariant)
    df["macd_std_pct"] = df["macd_std"] / df["Close"]
    df["macd_signal_std_pct"] = df["macd_signal_std"] / df["Close"]
    df["macd_hist_std_pct"] = df["macd_hist_std"] / df["Close"]
    # MACD diff changes
    df["macd_std_change"] = df["macd_std"].diff()
    df["macd_hist_std_change"] = df["macd_hist_std"].diff()

    # Slow MACD 1 month to 3 months
    macd_slow = macd(df["Close"], fast=21, slow=63, signal=15)
    macd_slow.columns = ["macd_slow", "macd_signal_slow", "macd_hist_slow"]
    df = pd.concat([df, macd_slow], axis=1)
    # MACD as % of price (scale-invariant)
    df["macd_slow_pct"] = df["macd_slow"] / df["Close"]
    df["macd_signal_slow_pct"] = df["macd_signal_slow"] / df["Close"]
    df["macd_hist_slow_pct"] = df["macd_hist_slow"] / df["Close"]
    # MACD diff changes
    df["macd_slow_change"] = df["macd_slow"].diff()
    df["macd_hist_slow_change"] = df["macd_hist_slow"].diff()

    # RSI - Multiple windows to capture momentum at different timescales
    for window in [5, 10, 21, 63]:
        df[f"rsi_{window}"] = rsi(df["Close"], window=window)
        df[f"rsi_{window}_change"] = df[f"rsi_{window}"].diff()

    # ROC - Multiple periods for different momentum horizons
    for period in [5, 10, 21, 63]:
        df[f"roc_{period}"] = roc(df["Close"], period=period)

    return df

In [9]:
# ============= 2. Volatility Features =============

def volatility_ft(df):
    # Pre-compute log returns once
    log_returns = np.log(df["Close"] / df["Close"].shift(1))
    
    # Rolling std of returns
    for wnd in [3, 5, 10, 21, 63]:
        df[f"vol_std_{wnd}"] = log_returns.rolling(window=wnd, min_periods=1).std()
                
    # Z-scores of volatility (regime detection)
    for wnd in [10, 21, 63]:
        mu = df[f"vol_std_{wnd}"].rolling(126, min_periods=63).mean()
        sd = df[f"vol_std_{wnd}"].rolling(126, min_periods=63).std()
        df[f"vol_std_zscore_{wnd}"] = (df[f"vol_std_{wnd}"] - mu) / (sd + 1e-10)

    # Rolling skewness of returns (asymmetric risk - negative skew = tail risk)
    for wnd in [21, 63]:
        df[f"return_skew_{wnd}"] = log_returns.rolling(window=wnd, min_periods=10).skew()

    # ATR - compute once per window, store both raw and pct
    for wnd in [3, 5, 10, 21, 63]:
        atr_val = atr(df["High"], df["Low"], df["Close"], window=wnd)
        df[f"atr_{wnd}"] = atr_val
        df[f"atr_{wnd}_pct_close"] = atr_val / df['Close']

    for wnd in [3, 5, 10, 21, 63]:
        atr_ema = ema(df[f"atr_{wnd}_pct_close"], wnd)
        df[f"atr_{wnd}_pct_close_dist_to_ema"] = (df[f"atr_{wnd}_pct_close"] - atr_ema) / (atr_ema.abs() + 1e-10)

    # Parkinson volatility - multiple windows
    for wnd in [3, 5, 10, 21, 63]:
        df[f"parkinson_vol_{wnd}"] = parkinson_vol(df["High"], df["Low"], window=wnd)

    # Bollinger Bands - multiple windows with % distance metrics
    for wnd in [3, 5, 10, 21, 63]:
        # Calculate SMA and standard deviation
        sma = df["Close"].rolling(window=wnd, min_periods=1).mean()
        std = df["Close"].rolling(window=wnd, min_periods=1).std()
        
        # Bollinger Bands (2 standard deviations)
        upper_band = sma + (2 * std)
        lower_band = sma - (2 * std)
        
        # % distance from bands (scale-invariant)
        df[f"bb_{wnd}_pct_from_upper"] = (df["Close"] - upper_band) / df["Close"]
        df[f"bb_{wnd}_pct_from_lower"] = (df["Close"] - lower_band) / df["Close"]

    return df

In [10]:
# ============= 3. Volume Features =============

def volume_ft(df):
    eps = 1e-10
    
    # Clean volume: replace 0 with NaN and ffill only
    # Zero volume days are treated as missing data (trading halts, no trades, data gaps)
    volume_clean = df["Volume"].replace(0, np.nan).ffill()
    
    # Volume pct change using cleaned volume
    for period in [1, 5, 10, 21]:
        df[f"vol_pct_change_{period}d"] = pct_change(volume_clean, period=period)

    # Volume Z-score at multiple windows
    for wnd in [10, 21, 63, 126]:
        vol_mean = volume_clean.rolling(window=wnd, min_periods=1).mean()
        vol_std = volume_clean.rolling(window=wnd, min_periods=1).std()
        df[f"vol_zscore_{wnd}"] = (volume_clean - vol_mean) / (vol_std + eps)

    # Volume ratio
    for wnd in [3, 5, 10, 21, 63]:
        vol_ma = volume_clean.rolling(wnd, min_periods=1).mean()
        df[f"vol_ratio_{wnd}"] = volume_clean / (vol_ma + eps)

    # Volume volatility
    for wnd in [5, 10, 21, 63]:
        df[f'volume_volatility_{wnd}'] = df["vol_pct_change_1d"].rolling(wnd, min_periods=1).std()

    # OBV - use original volume (0 volume = no contribution, which is correct)
    df["obv"] = obv(df["Close"], df["Volume"])

    for wnd in [5, 10, 21, 63]:
        obv_ema = ema(df["obv"], wnd)
        df[f"obv_dist_to_ema_{wnd}_pct"] = (df["obv"] - obv_ema) / (obv_ema.abs() + eps)

    # price alignment - use cleaned volume for denominator
    for wnd in [5, 10, 21, 63]:
        signed_vol_sum = df["obv"].diff(wnd)                       
        total_vol_sum = volume_clean.rolling(wnd, min_periods=1).sum()  
        df[f"vol_price_alignment_{wnd}"] = signed_vol_sum / (total_vol_sum + eps)

    # Stochastic %K and %D - multiple windows (already scale-invariant 0-100)
    for window in [5, 10, 21, 63, 126]:
        df[f"stoch_k_{window}"] = stoch_k(df["High"], df["Low"], df["Close"], window=window)
        df[f"stoch_d_{window}"] = stoch_d(df["High"], df["Low"], df["Close"], k_window=window, d_window=3)
        # %K - %D difference (useful for crossover signals)
        df[f"stoch_k_d_diff_{window}"] = df[f"stoch_k_{window}"] - df[f"stoch_d_{window}"]

    # Money Flow Multiplier - scale-invariant (-1 to +1)
    # Shows where close is relative to high-low range (buying/selling pressure)
    df["mfm"] = mfm(df["High"], df["Low"], df["Close"])
    
    # Rolling averages of MFM to smooth out noise
    for wnd in [3, 5, 10, 21, 63]:
        df[f"mfm_avg_{wnd}"] = df["mfm"].rolling(window=wnd, min_periods=1).mean()

    # A/D Oscillator - use original volume (0 = no contribution)
    df["ad_line"] = ad_oscillator(df["High"], df["Low"], df["Close"], df["Volume"])

    # A/D Z-score at multiple windows
    for wnd in [10, 21, 63, 126]:
        ad_mean = df["ad_line"].rolling(window=wnd, min_periods=1).mean()
        ad_std = df["ad_line"].rolling(window=wnd, min_periods=1).std()
        df[f"ad_zscore_{wnd}"] = (df["ad_line"] - ad_mean) / (ad_std + eps)
    
    # A/D line distance to EMA (scale-invariant: shows acceleration/deceleration)
    for wnd in [5, 10, 21, 63]:
        ad_ema = ema(df["ad_line"], wnd)
        df[f"ad_dist_to_ema_{wnd}_pct"] = (df["ad_line"] - ad_ema) / (ad_ema.abs() + eps)
    
    # Drop raw ad_line (not scale-invariant)
    df.drop(columns=["ad_line"], inplace=True)

    return df

In [11]:
# ============= 4. Price Structure Features =============

def price_structure_ft(df):
    # Basic ranges (absolute values)
    df["range_hl"] = df["High"] - df["Low"]
    df["body_size"] = (df["Close"] - df["Open"]).abs()
    df["upper_wick"] = df["High"] - pd.concat([df["Open"], df["Close"]], axis=1).max(axis=1)
    df["lower_wick"] = pd.concat([df["Open"], df["Close"]], axis=1).min(axis=1) - df["Low"]

    # Convert to percentages (scale-invariant)
    df["range_hl_pct"] = df["range_hl"] / df["Close"]
    df["body_size_pct"] = df["body_size"] / df["Close"]
    df["upper_wick_pct"] = df["upper_wick"] / df["Close"]
    df["lower_wick_pct"] = df["lower_wick"] / df["Close"]

    # Compression ratio (already scale-invariant)
    df["range_compression"] = df["body_size"] / (df["range_hl"] + 1e-10)

    # Gap features: Open vs previous Close (overnight information)
    # Positive gap = opened higher than previous close (bullish overnight sentiment)
    # Negative gap = opened lower than previous close (bearish overnight sentiment)
    prev_close = df["Close"].shift(1)
    df["gap_pct"] = (df["Open"] - prev_close) / (prev_close + 1e-10)

    # Drop absolute values, keep only percentages
    df.drop(columns=["range_hl", "body_size", "upper_wick", "lower_wick"], inplace=True)

    return df

In [12]:
# ============= 4b. Rolling 5-Day (Weekly) Price Structure Features =============
def weekly_price_structure_ft(df):
    # Calculate rolling 5-day (1 trading week) high, low, open (first), close (last)
    rolling_5d_high = df["High"].rolling(window=5, min_periods=1).max()
    rolling_5d_low = df["Low"].rolling(window=5, min_periods=1).min()
    # Use shift instead of slow lambda - gets the value from 4 days ago (start of 5-day window)
    rolling_5d_open = df["Open"].shift(4).fillna(df["Open"])
    rolling_5d_close = df["Close"]  # Current close is the "close" of the rolling period

    # Rolling 5-day price structure features (absolute values)
    weekly_range_hl = rolling_5d_high - rolling_5d_low
    weekly_body_size = (rolling_5d_close - rolling_5d_open).abs()
    
    # Use numpy maximum/minimum instead of pd.concat for speed
    weekly_upper_wick = rolling_5d_high - np.maximum(rolling_5d_open.values, rolling_5d_close.values)
    weekly_lower_wick = np.minimum(rolling_5d_open.values, rolling_5d_close.values) - rolling_5d_low

    # Convert to percentages (scale-invariant)
    df["weekly_range_hl_pct"] = weekly_range_hl / df["Close"]
    df["weekly_body_size_pct"] = weekly_body_size / df["Close"]
    df["weekly_upper_wick_pct"] = weekly_upper_wick / df["Close"]
    df["weekly_lower_wick_pct"] = weekly_lower_wick / df["Close"]

    # Compression ratio (already scale-invariant)
    df["weekly_range_compression"] = weekly_body_size / (weekly_range_hl + 1e-10)

    return df

In [13]:
# ============= 5. Calendar Features =============
def calendar_ft(df):
    # Get dates from index
    dates = pd.to_datetime(df.index, utc=True)

    # Day-of-week (0=Mon,...,4=Fri) - extracted from calendar date
    dow = dates.weekday
    df["dow"] = dow

    # One-hot for day-of-week (only Mon-Fri will have data)
    dow_dummies = pd.get_dummies(dow, prefix="dow", dtype=int)
    dow_dummies.index = df.index  # align
    df = pd.concat([df, dow_dummies], axis=1)

    # First/last week of month (based on calendar day)
    day = dates.day
    df["first_week_of_month"] = (day <= 7).astype(int)
    df["last_week_of_month"] = (day >= (dates + pd.offsets.MonthEnd(0)).day - 6).astype(int)

    return df

In [14]:
# ============= 6. Create Target Returns (Forward-Looking) =============
def create_targets_ft(df):
    # Create target variables: forward-looking LOG returns (shifted backward)
    # These align today's features with future returns for prediction
    # note these are in trading days
    df["target_ret_1d"] = np.log(df["Close"].shift(-1) / df["Close"])
    df["target_ret_3d"] = np.log(df["Close"].shift(-3) / df["Close"])
    df["target_ret_5d"] = np.log(df["Close"].shift(-5) / df["Close"])
    df["target_ret_10d"] = np.log(df["Close"].shift(-10) / df["Close"])
    df["target_ret_21d"] = np.log(df["Close"].shift(-21) / df["Close"])
    
    return df

In [15]:
# ============= 7. Cleanup Non-Scale-Invariant Columns =============
def cleanup_ft(df):
    # Filter to mature rows
    # Need 126+ trading days for: EMA 126, beta_126d, roc_63, etc.
    # Using 200 calendar days (~140 trading days) to be safe
    min_mature_days = 200
    min_mature_date = df.index.min() + pd.Timedelta(days=min_mature_days)
    df = df[df.index >= min_mature_date].copy()

    # Drop non-scale-invariant columns
    columns_to_drop = [
        # OHLCV (absolute prices/volumes)
        "Open", "High", "Low", "Close", "Volume",
        
        # Raw EMAs (absolute prices)
        "ema_3", "ema_5", "ema_10", "ema_21", "ema_63", "ema_126",
        
        # Absolute ATR values
        "atr_3", "atr_5", "atr_10", "atr_21", "atr_63",
        
        # OBV (absolute cumulative volume)
        "obv",
        
        # A/D line (absolute cumulative volume)
        "ad_line",
        
        # Raw MACD values (not scale-invariant, keeping % and % change versions)
        "macd_fast", "macd_signal_fast", "macd_hist_fast",
        "macd_std", "macd_signal_std", "macd_hist_std",
        "macd_slow", "macd_signal_slow", "macd_hist_slow",
        
        # Redundant one-hot encoded features
        "dow",  # We have dow_0 through dow_6 already
        "dow_0",  # Drop reference category to avoid multicollinearity (k-1 dummies)
    ]

    df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    
    # Identify target columns vs feature columns
    target_cols = [c for c in df.columns if c.startswith('target_')]
    feature_cols = [c for c in df.columns if c not in target_cols and c != 'ticker']
    
    # Check for NaNs in feature columns and ffill instead of dropping
    feature_nans = df[feature_cols].isna()
    rows_with_feature_nans = feature_nans.any(axis=1)
    
    if rows_with_feature_nans.any():
        nan_dates = df.index[rows_with_feature_nans].tolist()
        nan_cols = feature_nans.columns[feature_nans.any()].tolist()
        ticker_name = df['ticker'].iloc[0] if 'ticker' in df.columns else 'unknown'
        print(f"    [{ticker_name}] ffilling {len(nan_dates)} dates: {[d.date() for d in nan_dates[:5]]}{'...' if len(nan_dates) > 5 else ''}")
        print(f"      Columns with NaN: {nan_cols[:10]}{'...' if len(nan_cols) > 10 else ''}")
        
        # Forward fill NaNs in feature columns only
        df[feature_cols] = df[feature_cols].ffill()
        
        # If still NaN after ffill (e.g., NaN at start), drop those rows
        remaining_nans = df[feature_cols].isna().any(axis=1)
        if remaining_nans.any():
            n_drop = remaining_nans.sum()
            print(f"      Dropping {n_drop} rows with unfillable NaNs")
            df = df[~remaining_nans]
    
    # Target NaNs (last 21 rows) are kept - handled in training loop
    
    return df

In [16]:
# ============= Apply All Transformations to All Tickers =============
def apply_all_features(df, ticker_name, factor_returns):
    """Apply all feature transformations to a single ticker."""
    df = trend_ft(df, ticker_name)
    if df is None:
        return None
    df = volatility_ft(df)
    df = volume_ft(df)
    df = price_structure_ft(df)
    df = weekly_price_structure_ft(df)
    df = calendar_ft(df)
    df = beta_ft(df, factor_returns)  # Add beta feature
    df = create_targets_ft(df)
    df = cleanup_ft(df)
    return df

# Training

In [17]:
# Process all tickers with memory optimization and resume support
import gc
import os

# Process in batches and save incrementally
BATCH_SIZE = 50
OUTPUT_DIR = os.path.expanduser("stock_training_data_final")
# "/Users/gabriel/Library/CloudStorage/GoogleDrive-gdh@wharton.upenn.edu/My Drive/Sentiment Model/stock_training_data_final"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check which batches already exist for resume support
existing_batches = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.parquet')]
if existing_batches:
    # Find the highest batch number and get processed tickers
    processed_tickers = set()
    
    # Sort batches and exclude the last one (might be incomplete)
    sorted_batches = sorted(existing_batches)
    last_batch = sorted_batches[-1]
    complete_batches = sorted_batches[:-1]  # All except last
    
    # Load tickers from complete batches only
    for batch_file in complete_batches:
        batch_df = pd.read_parquet(os.path.join(OUTPUT_DIR, batch_file), columns=['ticker'])
        processed_tickers.update(batch_df['ticker'].unique())
        del batch_df
    
    # Delete the last batch file (will be reprocessed)
    os.remove(os.path.join(OUTPUT_DIR, last_batch))
    print(f"Deleted incomplete batch: {last_batch}")
    
    gc.collect()
    
    next_batch_num = int(last_batch.replace('batch_', '').replace('.parquet', ''))
    print(f"Resuming: {len(processed_tickers)} tickers already processed, restarting from batch {next_batch_num}")
else:
    processed_tickers = set()
    next_batch_num = 0
    print("Starting fresh processing")

failed_tickers = []

# Filter out already processed tickers - use sorted_tickers to maintain IPO date ordering
tickers = [t for t in sorted_tickers if t not in processed_tickers]
print(f"Tickers to process: {len(tickers)}")

if len(tickers) == 0:
    print("All tickers already processed!")
else:
    total_batches = (len(tickers) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_idx in tqdm(range(total_batches), desc="Processing batches"):
        batch_start = batch_idx * BATCH_SIZE
        batch_end = min(batch_start + BATCH_SIZE, len(tickers))
        batch_tickers = tickers[batch_start:batch_end]
        
        # Print batch date range information
        batch_first_dates = [ticker_first_dates[t] for t in batch_tickers]
        first_start_date = min(batch_first_dates).date()
        last_start_date = max(batch_first_dates).date()
        print(f"\n=== Batch {next_batch_num} Ticker Date Range ===")
        print(f"First ticker start date: {first_start_date}")
        print(f"Last ticker start date:  {last_start_date}")
        
        batch_results = []
        batch_failed = []  # Track failures within this batch
        
        for ticker in batch_tickers:
            try:
                ticker_df = ticker_data[ticker]
                result = apply_all_features(ticker_df, ticker, factor_returns)
                if result is not None and len(result) > 0:
                    batch_results.append(result)
                else:
                    batch_failed.append((ticker, "Insufficient data", len(ticker_df)))
                    failed_tickers.append((ticker, "Insufficient data"))
            except Exception as e:
                batch_failed.append((ticker, str(e), len(ticker_data[ticker])))
                failed_tickers.append((ticker, str(e)))
        
        # Print failed tickers in this batch with diagnostics
        if batch_failed:
            print(f"\n=== Batch {next_batch_num} Failed Tickers ===")
            for ticker, reason, n_rows in batch_failed:
                raw_df = ticker_data[ticker]
                print(f"  {ticker}: {reason}")
                print(f"    - Raw rows: {n_rows}")
                print(f"    - Date range: {raw_df.index.min().date()} to {raw_df.index.max().date()}")
                print(f"    - Trading days: {len(raw_df)}")
                print(f"    - Describe:")
                display(raw_df.describe())
        
        if batch_results:
            batch_df = pd.concat(batch_results, axis=0)
            batch_df = batch_df.reset_index().rename(columns={'index': 'date'})
            
            # NaN diagnostics before saving
            rows_with_nan = batch_df.isna().any(axis=1).sum()
            print(f"\n=== Batch {next_batch_num} NaN Summary ===")
            print(f"Rows: {batch_df.shape[0]:,}, Columns: {batch_df.shape[1]}")
            print(f"Rows with at least one NaN: {rows_with_nan:,}")
            print(f"Rows fully complete: {batch_df.shape[0] - rows_with_nan:,}")
            nan_per_col = batch_df.isna().sum()
            nan_cols = nan_per_col[nan_per_col > 0].sort_values(ascending=False)
            if len(nan_cols) > 0:
                print(f"NaNs per Column (non-zero only):")
                print(nan_cols)
            
            # Save batch as separate parquet file
            batch_file = os.path.join(OUTPUT_DIR, f'batch_{next_batch_num:04d}.parquet')
            batch_df.to_parquet(batch_file, index=False)
            next_batch_num += 1
            
            # Clear memory
            del batch_results, batch_df
            gc.collect()
        
        print(f"Batch {batch_idx + 1}/{total_batches} complete. Failed so far: {len(failed_tickers)}")

    print(f"\nProcessing complete!")
    print(f"Failed: {len(failed_tickers)} tickers")
    print(f"Output saved to: {OUTPUT_DIR}/")

Starting fresh processing
Tickers to process: 2910


Processing batches:   0%|          | 0/59 [00:00<?, ?it/s]


=== Batch 0 Ticker Date Range ===
First ticker start date: 2018-01-02
Last ticker start date:  2018-01-02

=== Batch 0 NaN Summary ===
Rows: 93,144, Columns: 175
Rows with at least one NaN: 1,050
Rows fully complete: 92,094
NaNs per Column (non-zero only):
target_ret_21d    1050
target_ret_10d     500
target_ret_5d      250
target_ret_3d      150
target_ret_1d       50
dtype: int64


Processing batches:   2%|▏         | 1/59 [00:04<03:56,  4.08s/it]

Batch 1/59 complete. Failed so far: 0

=== Batch 1 Ticker Date Range ===
First ticker start date: 2018-01-02
Last ticker start date:  2018-01-02

=== Batch 1 NaN Summary ===
Rows: 93,145, Columns: 175
Rows with at least one NaN: 1,050
Rows fully complete: 92,095
NaNs per Column (non-zero only):
target_ret_21d    1050
target_ret_10d     500
target_ret_5d      250
target_ret_3d      150
target_ret_1d       50
dtype: int64


...

,Open,High,Low,Close,Volume
count,155.000000,155.000000,155.000000,155.000000,1.550000e+02
mean,34.403374,34.732419,34.078097,34.388645,4.129129e+05
std,2.791369,2.578172,3.022374,2.826744,1.003642e+06
min,27.510000,28.094999,27.049999,27.510000,5.570000e+04
25%,31.780000,32.170000,31.045000,31.615000,1.440500e+05
50%,36.520000,36.660000,36.490002,36.500000,2.314000e+05
75%,36.759998,36.799999,36.689999,36.735001,3.495500e+05
max,37.189999,37.189999,37.060001,37.110001,1.033420e+07


  ETOR: Insufficient data
    - Raw rows: 152
    - Date range: 2025-05-13 to 2025-12-17
    - Trading days: 152
    - Describe:


,Open,High,Low,Close,Volume
count,152.000000,152.000000,152.000000,152.000000,1.520000e+02
mean,49.928257,51.265612,48.347132,49.715658,1.319181e+06
std,10.913173,11.537391,10.188071,10.860144,1.295761e+06
min,33.000000,34.669998,32.660000,33.470001,0.000000e+00
25%,40.429999,41.432500,39.339999,40.515000,7.210000e+05
50%,45.064999,46.402500,44.459999,44.880001,9.699000e+05
75%,61.125000,62.532501,59.055001,61.061249,1.458975e+06
max,76.639999,79.959999,66.209999,75.970001,1.307100e+07


Processing batches: 100%|██████████| 59/59 [03:29<00:00,  3.55s/it]


=== Batch 58 NaN Summary ===
Rows: 401, Columns: 175
Rows with at least one NaN: 168
Rows fully complete: 233
NaNs per Column (non-zero only):
target_ret_21d    168
target_ret_10d     80
target_ret_5d      40
target_ret_3d      24
target_ret_1d       8
dtype: int64
Batch 59/59 complete. Failed so far: 2

Processing complete!
Failed: 2 tickers
Output saved to: stock_training_data_final/


In [18]:
display(dropped_tickers)

['ABL',
 'ACEL',
 'AEBI',
 'AERO',
 'AFGE',
 'AHCO',
 'AKO-A',
 'ALH',
 'ALRS',
 'ALTI',
 'AMBQ',
 'AMCR',
 'AMRZ',
 'APXT',
 'ARX',
 'ASIC',
 'ATS',
 'AUGO',
 'BBOT',
 'BCAL',
 'BCSS',
 'BETA',
 'BETR',
 'BGM',
 'BGSI',
 'BHRB',
 'BITF',
 'BLLN',
 'BLSH',
 'BMNR',
 'BRBI',
 'BTBT',
 'BTDR',
 'BTQ',
 'BULL',
 'BWLP',
 'BWMX',
 'CAI',
 'CCCX',
 'CCZ',
 'CHYM',
 'CIG-C',
 'CLSK',
 'CMCSV',
 'CNL',
 'CPAC',
 'CRCL',
 'CRML',
 'CTOS',
 'CUBB',
 'DEC',
 'DMII',
 'DMIIU',
 'ECX',
 'EFC-PD',
 'EFXT',
 'EICA',
 'ELVR',
 'EMA',
 'ERO',
 'ESBA',
 'EVO',
 'EXEEL',
 'EXEEW',
 'EXEEZ',
 'FCRX',
 'FER',
 'FERG',
 'FIG',
 'FIGR',
 'FISK',
 'FLUT',
 'FLY',
 'FORTY',
 'FRMI',
 'FSUN',
 'GCMG',
 'GDYN',
 'GEGGL',
 'GIBO',
 'GLXY',
 'GRP-UN',
 'HBNB',
 'HCXY',
 'HDL',
 'HNGE',
 'HTFB',
 'HTFC',
 'HTFL',
 'HYMC',
 'IGIC',
 'IMTX',
 'INDV',
 'IVT',
 'JBS',
 'JCAP',
 'KDK',
 'KEN',
 'KLAR',
 'LGN',
 'LOT',
 'LUNR',
 'LWACU',
 'MAAS',
 'MH',
 'MHLA',
 'MIAX',
 'MICC',
 'MKC-V',
 'MLTX',
 'MNMD',
 'MNTN',
 'M

In [19]:
display(failed_tickers)

[('AHL', 'Insufficient data'), ('ETOR', 'Insufficient data')]

In [20]:
# # ============= 8. Verify Training Dataset =============
# # Data was already saved incrementally during processing

# # Load and verify the saved data
# final_df = pd.read_csv('stock_training_data.csv')
# print(f"Training dataset loaded from: stock_training_data.csv")
# print(f"Shape: {final_df.shape}")
# print(f"Unique tickers: {final_df['ticker'].nunique()}")
# print(f"Columns: {len(final_df.columns)}")
# display(final_df.head())

# Feature Engineering with Market Features

In [21]:
# ============= Market Features Setup (Independent) =============
import numpy as np
import pandas as pd
import os

# Load stock metadata
stock_meta_df = pd.read_csv('us_stocks_500m.csv')
print(f"Loaded stock metadata: {stock_meta_df.shape}")

# Get unique sectors, industries, countries
sectors = stock_meta_df['sector'].dropna().unique()
industries = stock_meta_df['industry'].dropna().unique()
countries = stock_meta_df['country'].dropna().unique()

print(f"Unique sectors: {len(sectors)}")
print(f"Unique industries: {len(industries)}")
print(f"Unique countries: {len(countries)}")

# Create random embeddings
np.random.seed(42)
sector_embeddings = {s: np.random.randn(6) for s in sectors}    # 11 sectors -> 6 dims
industry_embeddings = {i: np.random.randn(16) for i in industries}  # 148 industries -> 16 dims
country_embeddings = {c: np.random.randn(8) for c in countries}   # 50 countries -> 8 dims

Loaded stock metadata: (3098, 11)
Unique sectors: 11
Unique industries: 148
Unique countries: 50


In [22]:
def market_cap_bin(mcap_value):
    """
    Bin market cap into size categories.
    Returns dict with one-hot encoded values (micro as reference).
    """
    if pd.isna(mcap_value):
        label = 'unknown'
    elif mcap_value < 300_000_000:
        label = 'micro'
    elif mcap_value < 2_000_000_000:
        label = 'small'
    elif mcap_value < 10_000_000_000:
        label = 'mid'
    elif mcap_value < 200_000_000_000:
        label = 'large'
    else:
        label = 'mega'
    
    return {
        'mcap_small': 1 if label == 'small' else 0,
        'mcap_mid': 1 if label == 'mid' else 0,
        'mcap_large': 1 if label == 'large' else 0,
        'mcap_mega': 1 if label == 'mega' else 0,
    }

def get_market_features(row):
    """
    Get market features for a ticker row: sector/industry/country embeddings + mcap bin.
    Returns dict of features.
    """
    features = {'ticker': row['ticker']}
    
    # Sector embedding (6 dims)
    sector = row.get('sector')
    if pd.notna(sector) and sector in sector_embeddings:
        for i, val in enumerate(sector_embeddings[sector]):
            features[f'sector_emb_{i}'] = val
    else:
        for i in range(6):
            features[f'sector_emb_{i}'] = 0.0
    
    # Industry embedding (16 dims)
    industry = row.get('industry')
    if pd.notna(industry) and industry in industry_embeddings:
        for i, val in enumerate(industry_embeddings[industry]):
            features[f'industry_emb_{i}'] = val
    else:
        for i in range(16):
            features[f'industry_emb_{i}'] = 0.0
    
    # Country embedding (8 dims)
    country = row.get('country')
    if pd.notna(country) and country in country_embeddings:
        for i, val in enumerate(country_embeddings[country]):
            features[f'country_emb_{i}'] = val
    else:
        for i in range(8):
            features[f'country_emb_{i}'] = 0.0
    
    # Market cap bin (4 one-hot features, micro as reference)
    mcap = row.get('market_cap')
    mcap_features = market_cap_bin(mcap)
    features.update(mcap_features)
    
    return features

In [23]:
# ============= Create Market Features DataFrame (One Row Per Ticker) =============
market_features_list = []

for _, row in stock_meta_df.iterrows():
    features = get_market_features(row)
    market_features_list.append(features)

market_features_df = pd.DataFrame(market_features_list)

print(f"Market features shape: {market_features_df.shape}")
print(f"Columns: {list(market_features_df.columns)}")
display(market_features_df.head())

Market features shape: (3098, 35)
Columns: ['ticker', 'sector_emb_0', 'sector_emb_1', 'sector_emb_2', 'sector_emb_3', 'sector_emb_4', 'sector_emb_5', 'industry_emb_0', 'industry_emb_1', 'industry_emb_2', 'industry_emb_3', 'industry_emb_4', 'industry_emb_5', 'industry_emb_6', 'industry_emb_7', 'industry_emb_8', 'industry_emb_9', 'industry_emb_10', 'industry_emb_11', 'industry_emb_12', 'industry_emb_13', 'industry_emb_14', 'industry_emb_15', 'country_emb_0', 'country_emb_1', 'country_emb_2', 'country_emb_3', 'country_emb_4', 'country_emb_5', 'country_emb_6', 'country_emb_7', 'mcap_small', 'mcap_mid', 'mcap_large', 'mcap_mega']


,ticker,sector_emb_0,sector_emb_1,sector_emb_2,sector_emb_3,sector_emb_4,sector_emb_5,industry_emb_0,industry_emb_1,industry_emb_2,industry_emb_3,industry_emb_4,industry_emb_5,industry_emb_6,industry_emb_7,industry_emb_8,industry_emb_9,industry_emb_10,industry_emb_11,industry_emb_12,industry_emb_13,industry_emb_14,industry_emb_15,country_emb_0,country_emb_1,country_emb_2,country_emb_3,country_emb_4,country_emb_5,country_emb_6,country_emb_7,mcap_small,mcap_mid,mcap_large,mcap_mega
0,NVDA,0.496714,-0.138264,0.647689,1.52303,-0.234153,-0.234137,-0.072010,1.003533,0.361636,-0.645120,0.361396,1.538037,-0.035826,1.564644,-2.619745,0.821903,0.087047,-0.299007,0.091761,-1.987569,-0.219672,0.357113,0.225308,-0.369527,-0.131473,0.826047,-0.436764,-1.606577,1.749584,1.381454,0,0,0,1
1,AAPL,0.496714,-0.138264,0.647689,1.52303,-0.234153,-0.234137,1.477894,-0.518270,-0.808494,-0.501757,0.915402,0.328751,-0.529760,0.513267,0.097078,0.968645,-0.702053,-0.327662,-0.392108,-1.463515,0.296120,0.261055,0.225308,-0.369527,-0.131473,0.826047,-0.436764,-1.606577,1.749584,1.381454,0,0,0,1
2,GOOG,0.496714,-0.138264,0.647689,1.52303,-0.234153,-0.234137,0.005113,-0.234587,-1.415371,-0.420645,-0.342715,-0.802277,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,-1.918771,-0.026514,0.060230,2.463242,0.225308,-0.369527,-0.131473,0.826047,-0.436764,-1.606577,1.749584,1.381454,0,0,0,1
3,GOOGL,0.496714,-0.138264,0.647689,1.52303,-0.234153,-0.234137,0.005113,-0.234587,-1.415371,-0.420645,-0.342715,-0.802277,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,-1.918771,-0.026514,0.060230,2.463242,0.225308,-0.369527,-0.131473,0.826047,-0.436764,-1.606577,1.749584,1.381454,0,0,0,1
4,MSFT,0.496714,-0.138264,0.647689,1.52303,-0.234153,-0.234137,-0.192361,0.301547,-0.034712,-1.168678,1.142823,0.751933,0.791032,-0.909387,1.402794,-1.401851,0.586857,2.190456,-0.990536,-0.566298,0.099651,-0.503476,0.225308,-0.369527,-0.131473,0.826047,-0.436764,-1.606577,1.749584,1.381454,0,0,0,1


In [24]:
# ============= Save Market Features =============
csv_path = 'stock_market_features.csv'
market_features_df.to_csv(csv_path, index=False)

print(f"Market features saved to: {csv_path}")
print(f"Shape: {market_features_df.shape}")
print(f"Tickers: {market_features_df['ticker'].nunique()}")

Market features saved to: stock_market_features.csv
Shape: (3098, 35)
Tickers: 3098


# Testing

In [25]:
# # ---- CONFIG ----
# PARQUET_PATH = "stock_training_data_final/batch_0000.parquet"  # <- change this

# # ---- LOAD ----
# df = pd.read_parquet(PARQUET_PATH)

# # ---- BASIC INFO ----
# print("=== Columns ===")
# print(df.columns.tolist())

# print("\n=== Shape ===")
# print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")

# # ---- NaN DIAGNOSTICS ----
# rows_with_nan = df.isna().any(axis=1).sum()

# print("\n=== NaN Summary ===")
# print(f"Rows with at least one NaN: {rows_with_nan:,}")
# print(f"Rows fully complete: {df.shape[0] - rows_with_nan:,}")

# print("\n=== NaNs per Column (non-zero only) ===")
# nan_per_col = df.isna().sum()
# print(nan_per_col[nan_per_col > 0].sort_values(ascending=False))

In [26]:
# ============= Debug Ticker Data =============
DEBUG_TICKER = "ETOR"  # Change this to inspect different tickers
DATE_START = None # Optional: filter start date (None for no filter)
DATE_END = None   # Optional: filter end date (None for no filter)

if DEBUG_TICKER in ticker_data:
    raw_df = ticker_data[DEBUG_TICKER].copy()
    
    # Apply date filter if specified
    if DATE_START:
        raw_df = raw_df[raw_df.index >= pd.Timestamp(DATE_START, tz='UTC')]
    if DATE_END:
        raw_df = raw_df[raw_df.index <= pd.Timestamp(DATE_END, tz='UTC')]
    
    print(f"=== {DEBUG_TICKER} Raw Data ===")
    print(f"Shape: {raw_df.shape}")
    print(f"Date range: {raw_df.index.min().date()} to {raw_df.index.max().date()}")
    print(f"Trading days: {len(raw_df)}")
    
    print(f"\n=== Describe ===")
    display(raw_df.describe())
    
    print(f"\n=== Zero Volume Days ===")
    zero_vol = raw_df[raw_df['Volume'] == 0]
    print(f"Count: {len(zero_vol)}")
    if len(zero_vol) > 0:
        print(f"Zero-volume dates:")
        display(zero_vol)
    
    print(f"\n=== NaN Values in Raw Data ===")
    nan_rows = raw_df[raw_df.isna().any(axis=1)]
    print(f"Rows with NaN: {len(nan_rows)}")
    if len(nan_rows) > 0:
        display(nan_rows)
    
    print(f"\n=== Full Data for Date Range ===")
    display(raw_df)
else:
    print(f"Ticker {DEBUG_TICKER} not found in ticker_data")

=== ETOR Raw Data ===
Shape: (152, 6)
Date range: 2025-05-13 to 2025-12-17
Trading days: 152

=== Describe ===


,Open,High,Low,Close,Volume
count,152.000000,152.000000,152.000000,152.000000,1.520000e+02
mean,49.928257,51.265612,48.347132,49.715658,1.319181e+06
std,10.913173,11.537391,10.188071,10.860144,1.295761e+06
min,33.000000,34.669998,32.660000,33.470001,0.000000e+00
25%,40.429999,41.432500,39.339999,40.515000,7.210000e+05
50%,45.064999,46.402500,44.459999,44.880001,9.699000e+05
75%,61.125000,62.532501,59.055001,61.061249,1.458975e+06
max,76.639999,79.959999,66.209999,75.970001,1.307100e+07



=== Zero Volume Days ===
Count: 1
Zero-volume dates:


,ticker,Open,High,Low,Close,Volume
date,,,,,,
2025-05-13 04:00:00+00:00,ETOR,52.0,52.0,52.0,52.0,0



=== NaN Values in Raw Data ===
Rows with NaN: 0

=== Full Data for Date Range ===


,ticker,Open,High,Low,Close,Volume
date,,,,,,
2025-05-13 04:00:00+00:00,ETOR,52.000000,52.000000,52.000000,52.000000,0
2025-05-14 04:00:00+00:00,ETOR,69.690002,74.279999,65.059998,67.000000,13071000
2025-05-15 04:00:00+00:00,ETOR,65.150002,67.430000,60.759998,66.849998,3721300
2025-05-16 04:00:00+00:00,ETOR,66.589996,69.449997,62.439999,64.150002,2055300
2025-05-19 04:00:00+00:00,ETOR,61.590000,64.879997,61.403000,63.290001,884600
...,...,...,...,...,...,...
2025-12-11 05:00:00+00:00,ETOR,39.939999,40.650002,38.389999,39.270000,1341700
2025-12-12 05:00:00+00:00,ETOR,39.330002,39.480000,37.740002,37.950001,1265900
2025-12-15 05:00:00+00:00,ETOR,37.959999,38.269001,36.000000,36.549999,1322500


In [27]:
# ============= Liquidity Filter Function (for training loop) =============

def find_liquid_start(df, volume_col='Volume', min_liquid_pct=0.98, max_days_to_drop=252):
    """
    Find the earliest start date where the stock has >= min_liquid_pct non-zero volume days
    from that point forward. Drop up to max_days_to_drop (1 year) from the start.
    
    Call this in the training loop to get a clean df per ticker.
    
    Args:
        df: DataFrame with volume data (must have volume_col)
        volume_col: Name of the volume column
        min_liquid_pct: Minimum percentage of non-zero volume days required (default 98%)
        max_days_to_drop: Maximum trading days to drop from start (default 252 = 1 year)
    
    Returns:
        Trimmed DataFrame starting from liquid date, or None if can't achieve threshold
    """
    for days_to_drop in range(0, min(max_days_to_drop + 1, len(df))):
        trimmed = df.iloc[days_to_drop:]
        if len(trimmed) == 0:
            return None
        
        # Check if volume column exists
        if volume_col not in trimmed.columns:
            # If no volume column, assume it's already processed features
            # Check for vol_pct_change_1d as proxy
            if 'vol_pct_change_1d' in trimmed.columns:
                non_zero_pct = trimmed['vol_pct_change_1d'].notna().mean()
            else:
                return trimmed  # Can't check, return as-is
        else:
            non_zero_pct = (trimmed[volume_col] > 0).mean()
        
        if non_zero_pct >= min_liquid_pct:
            if days_to_drop > 0:
                print(f"      Dropped {days_to_drop} days to achieve {non_zero_pct:.1%} liquidity")
            return trimmed
    
    return None  # Couldn't achieve threshold even after dropping max days


# Test it
if DEBUG_TICKER in ticker_data:
    test_df = ticker_data[DEBUG_TICKER].copy()
    print(f"Original: {len(test_df)} rows, {(test_df['Volume'] > 0).mean():.1%} non-zero volume")
    
    result = find_liquid_start(test_df)
    if result is not None:
        print(f"After filter: {len(result)} rows, {(result['Volume'] > 0).mean():.1%} non-zero volume")
        print(f"New start date: {result.index.min().date()}")
    else:
        print("Could not achieve 98% liquidity threshold")

Original: 152 rows, 99.3% non-zero volume
After filter: 152 rows, 99.3% non-zero volume
New start date: 2025-05-13
